In [1]:
import os 
import sys
# using this for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))

import cv2
from src.utils.common import read_json


In [2]:
positions = read_json("../barbell_squat_rear_view_1_annotated_positions.json")

In [3]:
positions

{'NOSE': [[0.556820273399353, 0.3256584107875824],
  [0.5540304780006409, 0.32580843567848206],
  [0.5607544779777527, 0.3292604088783264],
  [0.5614351034164429, 0.3281187117099762],
  [0.5554317831993103, 0.3294084370136261],
  [0.5513877272605896, 0.32752475142478943],
  [0.5536937713623047, 0.32844942808151245],
  [0.5499846339225769, 0.32430070638656616],
  [0.5561969876289368, 0.32729387283325195],
  [0.5521190166473389, 0.32934775948524475],
  [0.5527427792549133, 0.3285926878452301],
  [0.5519381165504456, 0.3284679353237152],
  [0.5459152460098267, 0.3320068418979645],
  [0.5570425391197205, 0.32767462730407715],
  [0.5593671798706055, 0.3291408121585846],
  [0.547603189945221, 0.32999107241630554],
  [0.5503597259521484, 0.32939618825912476],
  [0.5549577474594116, 0.32749173045158386],
  [0.5415230393409729, 0.33081454038619995],
  [0.5281573534011841, 0.3276401460170746],
  [0.5289235711097717, 0.32632607221603394],
  [0.5291348695755005, 0.3269619643688202],
  [0.530291497

In [4]:
def interpolate_short_gaps_1d(arr, max_gap=3):
    """
    Linearly interpolate NaN gaps only if their length <= max_gap.
    Longer gaps remain NaN.

    Parameters
    ----------
    arr : array-like, shape (N,)
        1D signal with NaNs for missing values.
    max_gap : int
        Maximum consecutive NaN length to interpolate.

    Returns
    -------
    out : np.ndarray
        Interpolated copy.
    """
    out = np.asarray(arr, dtype=float).copy()
    n = len(out)

    isnan = np.isnan(out)
    if not isnan.any():
        return out

    i = 0
    while i < n:
        if not isnan[i]:
            i += 1
            continue

        start = i
        while i < n and isnan[i]:
            i += 1
        end = i  # first non-NaN after gap, or n

        gap_len = end - start

        left_idx = start - 1
        right_idx = end

        has_left = left_idx >= 0 and not np.isnan(out[left_idx])
        has_right = right_idx < n and not np.isnan(out[right_idx])

        # Only fill interior short gaps
        if gap_len <= max_gap and has_left and has_right:
            out[start:end] = np.interp(
                np.arange(start, end),
                [left_idx, right_idx],
                [out[left_idx], out[right_idx]]
            )

    return out

In [12]:
import numpy as np
x = [1,2,4,2,5, np.nan, np.nan, 4,6,10, np.nan, 9, np.nan, np.nan, np.nan, np.nan, 10, 13, np.nan]

In [13]:
x[1:3]

[2, 4]

In [ ]:
window = 2
for i in range(len(x)):
    v = x[i]
    if v is np.nan:
        gap_start = i 
        values_upper_idx = i
        values_lower_idx = values_upper_idx-(window)
        print(x[values_lower_idx:values_upper_idx])
        break

[2, 5]


In [ ]:
# linear interpolation


def linear_interpolation(arr, window = 2):
    """ 
    
    """
    out = np.asarray(arr, dtype=float).copy()
    n = len(out)

    isnan = np.isnan(out)
    if not isnan.any():
        return out


    gap_start_idx = None
    gap_end_idx = None
    currently_in_gap = False
    correction = None
    for i in range(n):
        v = out[i]
        # NOTE: values corresponding to gap_start and gap_end are within gap, so np.nan. 

        # checking if gap starts
        if np.isnan(v) and not currently_in_gap:
            # overwriting gap start
            currently_in_gap = True
            gap_start_idx = i

        # checking if next value is nan
        if np.isnan(v) and currently_in_gap:
            gap_end_idx = i 

        # checking if gap ended (interpolation here)
        if not np.isnan(v) and currently_in_gap:
            
            gap_length = gap_end_idx - gap_start_idx + 1
            
            # Computing mean starting values for gap interpolation
            left_side = out[max(0, (gap_start_idx-window)) : gap_start_idx]
            right_side = out[(gap_end_idx + 1 ) : min(n, gap_end_idx + window + 1)]

            # Edge case: Interpolating gap at beginning of array
            if len(left_side) == 0:
                # gap at the beginning: fill with nearest right value
                fill_value = right_side[0]
                correction = np.full(gap_length, fill_value)

            # Edge case: Interpolating gap at end of array
            if len(right_side) == 0:
                # gap at the end: fill with nearest left value
                fill_value = left_side[-1]
                correction = np.full(gap_length, fill_value)

            # default interpolation 
            if correction is None:
                # Edge case if left side or right side contain NAN values: 
                left_side = left_side[~np.isnan(left_side)]
                right_side = right_side[~np.isnan(right_side)]

                # computing correction by interpolation
                correction = np.interp(
                    [na_idx for na_idx in range(gap_start_idx, gap_end_idx+1)], # indices to fill
                    [max(0, gap_start_idx - 1), min((n-1), gap_end_idx+1)],      # anchor positions
                    [np.mean(left_side), np.mean(right_side)]
                )

            if len(correction) != gap_length:
                raise ValueError("Gap Length does not match correction length")
            out[gap_start_idx:gap_end_idx+1] = correction 
            # resetting counter-likes
            currently_in_gap = False
            gap_start_idx = None
            gap_end_idx = None
            correction = None

    return out

            





                








In [11]:
s = 2
e = 3

x =[i for i in range(s,e+1)]
x

[2, 3]